# Brent · records that may be consultation responses

**EDA exception to the golden rule, approved by the user on 2026-11-11.** In Lewisham,
some records turned out to be Lewisham's response when another authority consulted it on
that authority's own application (`findings/lewisham/out_of_borough.ipynb`). This
notebook checks Brent's records with the same decision values. Limits: read-only, only
the records being checked, only the fields needed to compare; nothing from here goes into
the bot, `semantic/`, tests or fixtures; only conclusions are written down.

Workflow: discuss the findings first. Brent's semantic files are **not** changed from
here; any confirmed ids go to `knowledge/lewisham_handover.md` for the API agent.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Brent's records with an opinion-type decision

Every Brent record whose decision is No Objection / Objection Raised to Proposal (OBS
only), Comment Issued or Closed.

In [2]:
OPINION = ["No Objection to Proposal (OBS only)", "Objection Raised to Proposal (OBS only)", "Comment Issued", "Closed"]
COLS = ["id", "lpa_name", "lpa_app_no", "status", "decision", "decision_date", "valid_date", "bo_system", "site_name", "description"]
body = {"size": 100, "_source": COLS, "sort": [{"id": "asc"}],
        "query": {"bool": {"filter": [brent_filter(), {"terms": {"decision.raw": OPINION}}]}}}
cand = pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]], columns=COLS)
print(len(cand), "records")
pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 300)
cand[["id", "lpa_name", "decision", "decision_date", "bo_system", "site_name", "description"]]

13 records


,id,lpa_name,decision,decision_date,bo_system,site_name,description
0,Brent-18_0592,Brent,No Objection to Proposal (OBS only),21/03/2018,CSV Data Import,"Garages rear of 1-30 AND 61-90 Barons Court, Church Lane, London",Prior approval application for demolition of garages rear of 1-30-...
1,Brent-18_0783,Brent,Objection Raised to Proposal (OBS only),04/04/2018,CSV Data Import,"5 Blackbird Hill, London, NW9 8RR",Prior notification for demolition of all redundant buildings assoc...
2,Brent-18_1482,Brent,No Objection to Proposal (OBS only),24/05/2018,CSV Data Import,"5 Blackbird Hill, London, NW9 8RR",Prior notification for demolition of all redundant buildings assoc...
3,Brent-18_2128,Brent,Objection Raised to Proposal (OBS only),17/07/2018,CSV Data Import,"All Units at 4-9 INC, and Garages rear of 4-9 Gladstone Parade, E...",Prior demolition notice for all buildings and rear garages
4,Brent-18_2452,Brent,No Objection to Proposal (OBS only),13/09/2018,CSV Data Import,"Interim Red Car Park, Royal Route, Wembley",Prior notification of proposed demolition of an existing ramp to t...
5,Brent-19_0775,Brent,No Objection to Proposal (OBS only),15/04/2019,CSV Data Import,"Queensbury Jet Wash Centre, Honeypot Lane, Stanmore, HA7 1EF",Prior notification for proposed demolition of buildings
6,OPDC-21_0044_DELBRE,OPDC,Closed,17/03/2021,Agile APAS,NaN,Prior notification of demolition of existing warehouse
7,OPDC-21_0075_DELBRE,OPDC,Closed,29/04/2021,Agile APAS,NaN,Redevelopment of the site for industrial purposes through the cons...
8,OPDC-21_0155_DELBRE,OPDC,Closed,15/09/2021,Agile APAS,NaN,Refurbishment and minor alterations to public house and garden (E(...
9,OPDC-21_0158_DELBRE,OPDC,Closed,22/09/2021,Agile APAS,Gatehouse,Demolition of existing gatehouse and installation of a replacement...


### Findings · Step 1

13 records: 6 Brent (CSV Data Import, 2018–2019), 7 OPDC (Agile APAS, 2021–2025).

- **The 6 Brent OBS-only records are prior notifications for demolition** (garages at
  Barons Court, 5 Blackbird Hill NW9 twice, Gladstone Parade, a ramp at the Wembley Red
  Car Park, Queensbury Jet Wash Centre on Honeypot Lane HA7). Brent's own decisions on
  demolition notices, not comments on another authority's application.
- **The 7 OPDC records** (Closed 6, Comment Issued 1) are OPDC's own cases: prior
  notification of demolition, redevelopment, a Section 73 variation, an EIA screening
  opinion. 4 have no site_name.

## Step 2 · Look for the original application in other authorities

For each record, search every other authority for a record with a real (not opinion-type)
decision, decided from 30 days before to 180 days after Brent's, whose `site_name`
contains the Brent record's postcode or its "house number + street". With only 13
records every match is listed for a person to review, with the word overlap of the two
descriptions.

In [3]:
import re

POSTCODE = re.compile(r"\b([A-Z]{1,2}\d[A-Z\d]?\s*\d[A-Z]{2})\b")
ADDRESS = re.compile(r"^(\d+[A-Z]?(?:-\d+[A-Z]?)?\s+[A-Z][A-Z' ]+)$")


def keys(site: object) -> list[str]:
    site = site.upper() if isinstance(site, str) else ""
    found = [m.group(1) for m in [POSTCODE.search(site)] if m]
    first = site.split(",")[0].strip()
    if ADDRESS.match(first):
        found.append(first)
    return found


def words(text: object) -> set[str]:
    return set(re.findall(r"[a-z]{4,}", text.lower())) if isinstance(text, str) else set()


rows = []
for r in cand.itertuples():
    ks = keys(r.site_name)
    if not ks or not isinstance(r.decision_date, str):
        rows.append({"brent_id": r.id, "key": "(no postcode or numbered address)" if not ks else "(no decision_date)"})
        continue
    d = pd.to_datetime(r.decision_date, format="%d/%m/%Y")
    window = {"range": {"decision_date": {"gte": (d - pd.Timedelta(days=30)).strftime("%d/%m/%Y"),
                                          "lte": (d + pd.Timedelta(days=180)).strftime("%d/%m/%Y"),
                                          "format": "dd/MM/yyyy"}}}
    for k in ks:
        # Read-only EDA lookup across other authorities (approved exception, see top).
        q = {"size": 10, "_source": ["lpa_name", "lpa_app_no", "decision", "decision_date", "site_name", "description"],
             "query": {"bool": {"filter": [window, {"match_phrase": {"site_name": k}}],
                                "must_not": [{"terms": {"lpa_name.raw": ["Brent", "OPDC"]}}, {"terms": {"decision.raw": OPINION}}]}}}
        hits = search(q)["hits"]["hits"]
        if not hits:
            rows.append({"brent_id": r.id, "key": k, "other_authority": "(none)"})
        for h in hits:
            o = h["_source"]
            a, b = words(r.description), words(o.get("description"))
            rows.append({"brent_id": r.id, "key": k, "other_authority": o.get("lpa_name"), "other_ref": o.get("lpa_app_no"),
                         "other_decision": o.get("decision"), "other_date": o.get("decision_date"),
                         "overlap": round(len(a & b) / len(a | b), 2) if a and b else None,
                         "other_site": (o.get("site_name") or "").replace("\n", " ").replace("\r", " ")[:45]})
pd.DataFrame(rows)

,brent_id,key,other_authority
0,Brent-18_0592,(no postcode or numbered address),NaN
1,Brent-18_0783,NW9 8RR,(none)
2,Brent-18_0783,5 BLACKBIRD HILL,(none)
3,Brent-18_1482,NW9 8RR,(none)
4,Brent-18_1482,5 BLACKBIRD HILL,(none)
5,Brent-18_2128,(no postcode or numbered address),NaN
6,Brent-18_2452,(no postcode or numbered address),NaN
7,Brent-19_0775,HA7 1EF,(none)
8,OPDC-21_0044_DELBRE,(no postcode or numbered address),NaN
9,OPDC-21_0075_DELBRE,(no postcode or numbered address),NaN


### Findings · Step 2

- **No match in any other authority.** Only 3 records have a postcode or numbered address
  (5 Blackbird Hill NW9 8RR, twice; Queensbury Jet Wash Centre HA7 1EF); none has a
  record in another authority decided around the same time. The other 10 cannot be
  looked up by address.
- **No consultation response found in Brent.** Together with Step 1 (the descriptions are
  Brent's and OPDC's own cases), there is nothing to drop. This is "none found", not
  proof that there are none: 10 records could not be looked up.
- The Queensbury Jet Wash Centre's address says Stanmore HA7; no other authority has a
  matching record, and a postcode or town alone does not show the borough.
- **Difference from Lewisham:** Brent uses OBS-only values for its own demolition
  notices; Lewisham uses them for its own opinions and for responses to neighbours.